# Customer Onboarding Analytics - Comprehensive Analysis

## Overview
This notebook performs end-to-end analysis of customer onboarding data for financial institutions.

**Key Areas:**
- Data exploration and quality checks
- Operational metrics calculation
- KYC compliance analysis
- Risk assessment and fraud detection
- Channel performance comparison
- Funnel analysis and conversion optimization

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Data Loading

In [ ]:
# Load datasets
applications = pd.read_csv('../data/raw/applications.csv')
kyc = pd.read_csv('../data/raw/kyc_verifications.csv')
events = pd.read_csv('../data/raw/onboarding_events.csv')
customers = pd.read_csv('../data/raw/customers.csv')

print(f"Applications: {len(applications):,} records")
print(f"KYC Verifications: {len(kyc):,} records")
print(f"Events: {len(events):,} records")
print(f"Customers: {len(customers):,} records")

## 2. Data Exploration

In [ ]:
# Applications dataset overview
print("=" * 80)
print("APPLICATIONS DATASET")
print("=" * 80)
print(f"\nShape: {applications.shape}")
print(f"\nColumns: {list(applications.columns)}")
print(f"\nData Types:\n{applications.dtypes}")
print(f"\nMissing Values:\n{applications.isnull().sum()[applications.isnull().sum() > 0]}")
applications.head()

In [ ]:
# Convert date columns
date_cols = ['application_date', 'processing_start_time', 'processing_end_time', 
             'approval_date', 'rejection_date']
for col in date_cols:
    if col in applications.columns:
        applications[col] = pd.to_datetime(applications[col])

events['event_timestamp'] = pd.to_datetime(events['event_timestamp'])

print("✓ Date columns converted")

## 3. Operational Metrics

In [ ]:
# Application status breakdown
status_counts = applications['application_status'].value_counts()
status_pct = applications['application_status'].value_counts(normalize=True) * 100

status_df = pd.DataFrame({
    'Count': status_counts,
    'Percentage': status_pct
})

print("APPLICATION STATUS DISTRIBUTION")
print("=" * 50)
print(status_df)

# Visualize
fig = px.pie(values=status_counts.values, names=status_counts.index, 
             title='Application Status Distribution',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

In [ ]:
# Processing time analysis
processed = applications[applications['processing_duration_hours'].notna()]

print("PROCESSING TIME STATISTICS")
print("=" * 50)
print(f"Count: {len(processed):,}")
print(f"Mean: {processed['processing_duration_hours'].mean():.2f} hours")
print(f"Median: {processed['processing_duration_hours'].median():.2f} hours")
print(f"Std Dev: {processed['processing_duration_hours'].std():.2f} hours")
print(f"Min: {processed['processing_duration_hours'].min():.2f} hours")
print(f"Max: {processed['processing_duration_hours'].max():.2f} hours")
print(f"\nPercentiles:")
print(f"25th: {processed['processing_duration_hours'].quantile(0.25):.2f} hours")
print(f"50th: {processed['processing_duration_hours'].quantile(0.50):.2f} hours")
print(f"75th: {processed['processing_duration_hours'].quantile(0.75):.2f} hours")
print(f"95th: {processed['processing_duration_hours'].quantile(0.95):.2f} hours")

# SLA compliance (48 hours)
within_sla = (processed['processing_duration_hours'] <= 48).sum()
sla_rate = within_sla / len(processed) * 100
print(f"\nSLA Compliance (48h): {sla_rate:.1f}% ({within_sla:,}/{len(processed):,})")

In [ ]:
# Processing time visualization
fig = px.histogram(processed, x='processing_duration_hours', nbins=50,
                   title='Processing Time Distribution',
                   labels={'processing_duration_hours': 'Processing Time (hours)'})
fig.add_vline(x=48, line_dash="dash", line_color="red", 
              annotation_text="SLA: 48 hours")
fig.show()

## 4. KYC Compliance Analysis

In [ ]:
# KYC success rates by document type
kyc_success = kyc.groupby('document_type').agg({
    'verification_status': [
        ('total', 'count'),
        ('verified', lambda x: (x == 'verified').sum()),
        ('failed', lambda x: (x == 'failed').sum())
    ]
}).round(2)

kyc_success.columns = ['total', 'verified', 'failed']
kyc_success['success_rate'] = (kyc_success['verified'] / kyc_success['total'] * 100).round(2)
kyc_success = kyc_success.sort_values('success_rate', ascending=False)

print("KYC SUCCESS RATES BY DOCUMENT TYPE")
print("=" * 70)
print(kyc_success)

# Visualize
fig = px.bar(kyc_success.reset_index(), x='document_type', y='success_rate',
             title='KYC Verification Success Rate by Document Type',
             labels={'success_rate': 'Success Rate (%)', 'document_type': 'Document Type'},
             color='success_rate', color_continuous_scale='RdYlGn')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Quality scores analysis
print("QUALITY SCORES")
print("=" * 50)
print(f"Document Quality: {kyc['document_quality_score'].mean():.3f}")
print(f"OCR Confidence: {kyc['ocr_confidence_score'].mean():.3f}")
print(f"Face Match: {kyc['face_match_score'].mean():.3f}")
print(f"\nAverage Verification Attempts: {kyc['verification_attempts'].mean():.2f}")

# Quality score by verification status
quality_by_status = kyc.groupby('verification_status')[[
    'document_quality_score', 'ocr_confidence_score', 'face_match_score'
]].mean()

print("\nQuality Scores by Verification Status:")
print(quality_by_status)

In [ ]:
# Top failure reasons
failed_kyc = kyc[kyc['verification_status'].isin(['failed', 'rejected'])]
failure_reasons = failed_kyc['failure_reason'].value_counts().head(10)

print("TOP 10 KYC FAILURE REASONS")
print("=" * 50)
for reason, count in failure_reasons.items():
    pct = count / len(failed_kyc) * 100
    print(f"{reason}: {count:,} ({pct:.1f}%)")

# Visualize
fig = px.bar(x=failure_reasons.values, y=failure_reasons.index, orientation='h',
             title='Top 10 KYC Failure Reasons',
             labels={'x': 'Count', 'y': 'Failure Reason'})
fig.show()

## 5. Risk and Fraud Analysis

In [ ]:
# Risk distribution
risk_dist = applications['risk_category'].value_counts()
risk_pct = applications['risk_category'].value_counts(normalize=True) * 100

print("RISK DISTRIBUTION")
print("=" * 50)
for category in ['low', 'medium', 'high']:
    count = risk_dist.get(category, 0)
    pct = risk_pct.get(category, 0)
    print(f"{category.capitalize()}: {count:,} ({pct:.1f}%)")

print(f"\nAverage Risk Score: {applications['risk_score'].mean():.3f}")
print(f"Median Risk Score: {applications['risk_score'].median():.3f}")

In [ ]:
# Fraud analysis
fraud_count = applications['fraud_flag'].sum()
fraud_rate = fraud_count / len(applications) * 100

print("FRAUD STATISTICS")
print("=" * 50)
print(f"Fraud Cases: {fraud_count:,}")
print(f"Fraud Rate: {fraud_rate:.2f}%")

# Fraud by risk category
fraud_by_risk = applications.groupby('risk_category').agg({
    'fraud_flag': ['sum', 'count']
})
fraud_by_risk.columns = ['fraud_cases', 'total']
fraud_by_risk['fraud_rate'] = (fraud_by_risk['fraud_cases'] / fraud_by_risk['total'] * 100).round(2)

print("\nFraud Rate by Risk Category:")
print(fraud_by_risk)

In [ ]:
# Risk score distribution by status
fig = px.box(applications, x='application_status', y='risk_score',
             title='Risk Score Distribution by Application Status',
             color='application_status')
fig.show()

## 6. Channel Performance Analysis

In [ ]:
# Channel performance
channel_perf = applications.groupby('channel').agg({
    'application_id': 'count',
    'application_status': lambda x: (x == 'approved').sum() / len(x) * 100,
    'processing_duration_hours': 'mean',
    'fraud_flag': 'sum',
    'risk_score': 'mean'
}).round(2)

channel_perf.columns = ['applications', 'approval_rate', 'avg_processing_hrs', 
                        'fraud_cases', 'avg_risk_score']

print("CHANNEL PERFORMANCE")
print("=" * 80)
print(channel_perf)

# Device performance
device_perf = applications.groupby('device_type').agg({
    'application_id': 'count',
    'application_status': lambda x: (x == 'approved').sum() / len(x) * 100
}).round(2)

device_perf.columns = ['applications', 'approval_rate']

print("\nDEVICE PERFORMANCE")
print("=" * 50)
print(device_perf)

In [ ]:
# Channel comparison visualization
fig = px.bar(channel_perf.reset_index(), x='channel', y=['applications', 'approval_rate'],
             title='Channel Performance Comparison',
             barmode='group')
fig.show()

## 7. Funnel Analysis

In [ ]:
# Calculate funnel
event_counts = events.groupby('event_type')['application_id'].nunique()

funnel_stages = [
    'application_started',
    'personal_details_completed',
    'document_uploaded',
    'kyc_completed',
    'terms_accepted',
    'account_created'
]

funnel_data = []
prev_count = None

for stage in funnel_stages:
    if stage in event_counts.index:
        count = event_counts[stage]
        if prev_count:
            conversion = count / prev_count * 100
            drop_off = 100 - conversion
        else:
            conversion = 100.0
            drop_off = 0.0
        
        funnel_data.append({
            'Stage': stage.replace('_', ' ').title(),
            'Count': count,
            'Conversion': f"{conversion:.1f}%",
            'Drop-off': f"{drop_off:.1f}%"
        })
        prev_count = count

funnel_df = pd.DataFrame(funnel_data)

print("ONBOARDING FUNNEL")
print("=" * 80)
print(funnel_df.to_string(index=False))

In [ ]:
# Funnel visualization
fig = go.Figure(go.Funnel(
    y=[row['Stage'] for row in funnel_data],
    x=[row['Count'] for row in funnel_data],
    textinfo="value+percent initial"
))

fig.update_layout(title='Onboarding Funnel')
fig.show()

## 8. Time-based Analysis

In [ ]:
# Daily trend
applications['date'] = applications['application_date'].dt.date
daily_apps = applications.groupby('date').size().reset_index(name='count')
daily_apps['date'] = pd.to_datetime(daily_apps['date'])
daily_apps['ma7'] = daily_apps['count'].rolling(window=7).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_apps['date'], y=daily_apps['count'], 
                         mode='lines', name='Daily',
                         line=dict(color='lightblue')))
fig.add_trace(go.Scatter(x=daily_apps['date'], y=daily_apps['ma7'],
                         mode='lines', name='7-day MA',
                         line=dict(color='darkblue', width=2)))
fig.update_layout(title='Daily Application Volume', xaxis_title='Date',
                 yaxis_title='Applications')
fig.show()

In [ ]:
# Day of week pattern
applications['day_of_week'] = applications['application_date'].dt.day_name()
dow_counts = applications['day_of_week'].value_counts()

# Order by weekday
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_ordered = dow_counts.reindex(day_order)

fig = px.bar(x=dow_ordered.index, y=dow_ordered.values,
             title='Applications by Day of Week',
             labels={'x': 'Day', 'y': 'Count'})
fig.show()

In [ ]:
# Hour of day pattern
applications['hour'] = applications['application_date'].dt.hour
hourly_counts = applications['hour'].value_counts().sort_index()

fig = px.bar(x=hourly_counts.index, y=hourly_counts.values,
             title='Applications by Hour of Day',
             labels={'x': 'Hour', 'y': 'Count'})
fig.show()

## 9. Geographic Analysis

In [ ]:
# Top states
state_stats = applications.groupby('state').agg({
    'application_id': 'count',
    'application_status': lambda x: (x == 'approved').sum() / len(x) * 100,
    'fraud_flag': 'sum'
}).round(2)

state_stats.columns = ['applications', 'approval_rate', 'fraud_cases']
state_stats = state_stats.sort_values('applications', ascending=False).head(15)

print("TOP 15 STATES")
print("=" * 70)
print(state_stats)

# Visualize
fig = px.bar(state_stats.reset_index(), x='state', y='applications',
             title='Top 15 States by Application Volume',
             color='approval_rate', color_continuous_scale='RdYlGn')
fig.show()

## 10. Key Insights & Recommendations

In [ ]:
print("=" * 80)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("=" * 80)

# Calculate key metrics
total_apps = len(applications)
approval_rate = (applications['application_status'] == 'approved').sum() / total_apps * 100
avg_processing = processed['processing_duration_hours'].mean()
kyc_success_rate = (kyc['verification_status'] == 'verified').sum() / len(kyc) * 100
fraud_rate = applications['fraud_flag'].sum() / total_apps * 100

print(f"\n1. OPERATIONAL EFFICIENCY")
print(f"   • Approval Rate: {approval_rate:.1f}%")
print(f"   • Avg Processing Time: {avg_processing:.1f} hours")
print(f"   • SLA Compliance: {sla_rate:.1f}%")
print(f"   💡 Recommendation: Focus on reducing processing time for {100-sla_rate:.1f}% of cases exceeding 48h SLA")

print(f"\n2. KYC COMPLIANCE")
print(f"   • Overall Success Rate: {kyc_success_rate:.1f}%")
print(f"   • Manual Review Required: {(kyc['manual_review_required'].sum() / len(kyc) * 100):.1f}%")
print(f"   💡 Recommendation: Improve document quality guidance to reduce manual reviews")

print(f"\n3. FRAUD DETECTION")
print(f"   • Fraud Rate: {fraud_rate:.2f}%")
print(f"   • High Risk Applications: {(applications['risk_category'] == 'high').sum() / total_apps * 100:.1f}%")
print(f"   💡 Recommendation: Enhance screening for high-risk categories")

print(f"\n4. CHANNEL OPTIMIZATION")
top_channel = applications['channel'].value_counts().index[0]
print(f"   • Leading Channel: {top_channel.capitalize()}")
print(f"   💡 Recommendation: Optimize mobile experience as it's the primary channel")

print(f"\n5. CONVERSION FUNNEL")
if len(funnel_data) > 0:
    overall_conversion = funnel_data[-1]['Count'] / funnel_data[0]['Count'] * 100
    print(f"   • Overall Conversion: {overall_conversion:.1f}%")
    print(f"   💡 Recommendation: Address drop-off points in document upload and KYC stages")

print("\n" + "=" * 80)

## Conclusion

This analysis provides comprehensive insights into the customer onboarding process, highlighting:
- Operational efficiency metrics
- KYC compliance performance
- Risk and fraud patterns
- Channel effectiveness
- Conversion funnel opportunities

These insights can drive data-informed decisions to optimize the onboarding experience and improve conversion rates.